In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from pathlib import Path

RESULTS_DIR = Path("../results")
BACKBONES = ["dinov2", "resnet50", "resnet18", "vgg16"]
METRICS = [
    "mean_cost",
    "sinkhorn_euclidean",
    "sinkhorn_cosine",
    "gaussian_mmd",
    "centroid_distance",
]

MODALITY_TO_TAX = {
    "SAR-QXSLAB": "SAR",
    "SAR-SEN12": "SAR",
    "OPT-QXSLAB": "OPT",
    "OPT-SEN12": "OPT",
    "CT": "CT",
    "T1": "T1T2",
    "T2": "T1T2",
    "HE": "HE",
    "IHC": "IHC",
    "ImageNet-1k": None,
}


TAXA = ["SAR", "OPT", "CT", "T1T2", "HE", "IHC"]
T_SCORE_RAW = [
    [4, 1, 0, 0, 0, 0],  # SAR
    [1, 4, 1, 1, 3, 3],  # OPT
    [0, 1, 4, 4, 1, 1],  # CT
    [0, 1, 4, 4, 1, 1],  # T1T2
    [0, 3, 1, 1, 4, 4],  # HE
    [0, 3, 1, 1, 4, 4],  # IHC
    # SAR, OPT, CT, T1T2, HE, IHC
]
T_SCORE_DF = pd.DataFrame(T_SCORE_RAW, index=TAXA, columns=TAXA)

In [8]:
def get_pairs(dist_df):
    """Return list of (distance, t_score) for all upper-triangle off-diagonal pairs
    where both modalities have a known taxonomy category."""
    mods = list(dist_df.index)
    same, cross = [], []
    for i in range(len(mods)):
        for j in range(i + 1, len(mods)):
            tax_i = MODALITY_TO_TAX.get(mods[i])
            tax_j = MODALITY_TO_TAX.get(mods[j])
            if tax_i is None or tax_j is None:
                continue
            t_score = T_SCORE_DF.loc[tax_i, tax_j]
            d = dist_df.iloc[i, j]
            if t_score == 4:
                same.append(d)
            else:
                cross.append(d)
    return np.array(cross), np.array(same)


rows = []
for backbone in BACKBONES:
    for metric in METRICS:
        path = RESULTS_DIR / backbone / f"{metric}_mean.csv"
        dist_df = pd.read_csv(path, index_col=0)
        cross, same = get_pairs(dist_df)
        stat, p = mannwhitneyu(cross, same, alternative="greater")
        rows.append(
            {
                "backbone": backbone,
                "metric": metric,
                "n_cross": len(cross),
                "n_same": len(same),
                "median_cross": np.median(cross),
                "median_same": np.median(same),
                "U_statistic": stat,
                "p_value": p,
                "significant": p < 0.05,
            }
        )

results_df = pd.DataFrame(rows)
results_df

,backbone,metric,n_cross,n_same,median_cross,median_same,U_statistic,p_value,significant
0,dinov2,mean_cost,30,6,60.327753,44.405225,166.0,1.991999e-04,True
1,dinov2,sinkhorn_euclidean,30,6,3364.402925,1438.947836,177.0,3.593813e-06,True
2,dinov2,sinkhorn_cosine,30,6,0.752889,0.324067,171.0,4.620617e-05,True
3,dinov2,gaussian_mmd,30,6,0.724447,0.415417,179.0,1.026804e-06,True
4,dinov2,centroid_distance,30,6,45.115702,21.318177,180.0,5.134018e-07,True
5,resnet50,mean_cost,30,6,19.386917,15.750859,174.0,1.540206e-05,True
6,resnet50,sinkhorn_euclidean,30,6,355.627653,196.038765,177.0,3.593813e-06,True
7,resnet50,sinkhorn_cosine,30,6,0.825619,0.408389,178.0,2.053607e-06,True
8,resnet50,gaussian_mmd,30,6,0.625931,0.336758,169.0,8.676491e-05,True
9,resnet50,centroid_distance,30,6,12.862732,6.922867,177.0,3.593813e-06,True


In [9]:
sig = results_df[results_df["significant"]]
print(f"  {len(sig)} / {len(results_df)} tests support p<0.05")

  19 / 20 tests support p<0.05
